# Lab 06: Tracing & Reproducible Agents

## Overview

In this lab you will **instrument your LangChain agent with MLflow tracing**, inspect individual spans in the Experiment UI, attach reproducibility tags to every run, and compare agent versions programmatically using `mlflow.search_runs`.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Application Development (30%) |
| **Key Skills** | MLflow autologging, trace inspection, experiment tagging, run comparison |
| **Estimated Time** | 25 minutes |
| **Estimated Cost** | ~$1 |

### What You Will Build

```
Agent Query
    └► AgentExecutor  (rebuilt from Lab 05)
            └► mlflow.langchain.autolog(log_traces=True)
                    ├► Span: LLM Call        (prompt tokens, latency)
                    ├── Span: Tool Call      (tool name, input/output)
                    └── Span: Retrieval      (query, num_results, docs)
                            └► MLflow Experiment UI  (trace timeline)

mlflow.start_run + set_tags
    └► tags: agent_version, llm_endpoint, embedding_model,
             chunk_size, chunk_overlap, vector_search_type
                    └► mlflow.search_runs  →  comparison table
```

Tracing is the primary observability primitive for agents. Every tool invocation, LLM call, and retrieval step produces a **span** — a timestamped record of inputs, outputs, and latency. Tagging runs lets you reproduce any experiment configuration and compare it against future versions.

Continue to **`workbook.md`** in this directory for the architecture diagram, concept map, and exam-style practice questions.

In [ ]:
# Prerequisites — run once per session
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-community unitycatalog-langchain databricks-agents --quiet
dbutils.library.restartPython()

> **Cost tip:** These labs run on **serverless compute** by default — no cluster setup needed. You only pay per-second of actual usage. The `COST_PROFILE` below lets you choose cheaper models if you're cost-sensitive.

In [ ]:
# === Cost Profile ===
# "budget"   — cheapest models, may have lower quality (~$0.50/lab)
# "balanced" — good quality/cost ratio (~$1-2/lab)  [DEFAULT]
# "quality"  — best models, highest cost (~$3-5/lab)
COST_PROFILE = "balanced"

_LLM_ENDPOINTS = {
    "budget":   "databricks-meta-llama-3-1-8b-instruct",
    "balanced": "databricks-meta-llama-3-3-70b-instruct",
    "quality":  "databricks-meta-llama-3-3-70b-instruct",
}
LLM_ENDPOINT = _LLM_ENDPOINTS[COST_PROFILE]

# ---------------------------------------------------------------------------
# Configuration — update these values to match your environment
# ---------------------------------------------------------------------------
CATALOG      = "genai_lab_guide"               # Unity Catalog catalog name
SCHEMA       = "default"                        # Schema inside the catalog
VS_ENDPOINT  = "genai_lab_guide_vs_endpoint"   # Vector Search endpoint name
VS_INDEX     = f"{CATALOG}.{SCHEMA}.arxiv_chunks_index"

# Agent version label — increment this each time you change the configuration
AGENT_VERSION = "v1"

print(f"Catalog      : {CATALOG}")
print(f"Schema       : {SCHEMA}")
print(f"VS Index     : {VS_INDEX}")
print(f"Agent version: {AGENT_VERSION}")

# ---------------------------------------------------------------------------
# Rebuild the agent from Lab 05
# These components were built in Lab 05; we reconstruct them here so this
# notebook runs standalone.  See Lab 05 for full implementation details.
# ---------------------------------------------------------------------------
from databricks.vector_search.client import VectorSearchClient
from langchain_community.chat_models import ChatDatabricks
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

# ── Retrieval tool (Lab 03) ───────────────────────────────────────────────────
vsc   = VectorSearchClient()
index = vsc.get_index(VS_ENDPOINT, VS_INDEX)


def retrieve_context(query: str, num_results: int = 3) -> str:
    results = index.similarity_search(
        query_text=query,
        columns=["chunk_text", "path"],
        num_results=num_results,
        query_type="hybrid",
    )
    docs = results.get("result", {}).get("data_array", [])
    parts = []
    for doc in docs:
        source = doc[1].split("/")[-1].replace(".pdf", "")
        parts.append(f"[Source: {source}]\n{doc[0]}")
    return "\n\n---\n\n".join(parts)


@tool
def search_arxiv_papers(query: str) -> str:
    """Search the arXiv paper corpus for context relevant to the user's question.
    Always call this tool before answering research questions."""
    return retrieve_context(query)


# ── UC function tools (Lab 04) ────────────────────────────────────────────────
client = DatabricksFunctionClient()
uc_toolkit = UCFunctionToolkit(
    function_names=[
        f"{CATALOG}.{SCHEMA}.get_paper_metadata",
        f"{CATALOG}.{SCHEMA}.format_citation",
    ],
    client=client,
)
uc_tools  = uc_toolkit.tools
all_tools = [search_arxiv_papers] + uc_tools

# ── LLM ──────────────────────────────────────────────────────────────────────
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    max_tokens=1024,
    temperature=0.1,
)

# ── Agent ─────────────────────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an arXiv Research Assistant. "
        "Use search_arxiv_papers to retrieve relevant passages before answering. "
        "Use get_paper_metadata for statistics and format_citation for references.",
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent          = create_tool_calling_agent(llm, all_tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=all_tools, verbose=False)

print(f"Tools    : {[t.name for t in all_tools]}")
print("Agent rebuilt successfully.")

## A. Enable MLflow Tracing

**MLflow tracing** is enabled with a single call to `mlflow.langchain.autolog(log_traces=True)`. Once active, every `AgentExecutor.invoke()` call automatically produces a **trace** — a tree of timestamped **spans** that records:

- The top-level agent invocation (inputs, outputs, total latency)
- Each LLM call (prompt, completion, token counts)
- Each tool call (tool name, input arguments, output)
- Each retrieval step (query text, returned documents)

Traces are stored in the **MLflow Experiment** identified by `mlflow.set_experiment()`. They appear in the Experiment UI under the **Traces** tab and can also be queried programmatically with `mlflow.search_traces()`.

> **No code changes to the agent are required.** Autologging intercepts LangChain callbacks transparently.

In [ ]:
import mlflow

# ── Enable LangChain autologging with tracing ─────────────────────────────────
# log_traces=True  : emit a trace for every chain/agent invocation
# log_models=False : we will log the model explicitly in Section D
mlflow.langchain.autolog(log_traces=True, log_models=False)

# ── Set the MLflow experiment ─────────────────────────────────────────────────
# Use your personal user path to avoid collisions in shared workspaces.
# Replace <your-username> with your actual Databricks user name.
USERNAME = spark.sql("SELECT current_user()").collect()[0][0]
EXPERIMENT_PATH = f"/Users/{USERNAME}/genai-lab-guide/lab-06-tracing"

experiment = mlflow.set_experiment(EXPERIMENT_PATH)

print(f"Tracing enabled : True")
print(f"Experiment path : {EXPERIMENT_PATH}")
print(f"Experiment ID   : {experiment.experiment_id}")
print()
print("All agent invocations in this notebook will now emit traces.")
print("Navigate to the Experiments tab in the left sidebar to see them.")

## B. Run Queries and Inspect Traces

With tracing enabled, every `agent_executor.invoke()` call silently emits a trace to the active MLflow experiment. The trace is visible in the **MLflow Experiment UI** immediately after the call completes.

**To inspect a trace in the UI:**
1. Open the **Experiments** tab in the Databricks left sidebar.
2. Select the experiment at `{EXPERIMENT_PATH}`.
3. Click the **Traces** tab (next to Runs).
4. Click any row to open the trace timeline.
5. Expand individual spans to see inputs, outputs, and timing.

Each query below exercises a different combination of tools so you can compare the span trees.

In [ ]:
# Run three queries — each produces a separate trace in the MLflow Experiment UI.
# After running this cell, navigate to the Traces tab to see the span trees.

queries = [
    "What is the key idea behind the transformer attention mechanism?",
    "Format a citation for Devlin et al. 2018, BERT, arXiv:1810.04805.",
    "How many chunks do we have indexed for the LoRA paper?",
]

for i, query in enumerate(queries, 1):
    print(f"\n{'=' * 70}")
    print(f"Query {i}: {query}")
    result = agent_executor.invoke({"input": query})
    print(f"Answer  : {result['output'][:300]}")

print("\nAll queries complete.")
print("Open the MLflow Experiment UI → Traces tab to inspect span trees.")
print(f"Experiment: {EXPERIMENT_PATH}")

## C. Inspect Trace Details Programmatically

`mlflow.search_traces()` returns a **Pandas DataFrame** where each row is one trace. Key columns include:

| Column | Description |
|---|---|
| `request_id` | Unique trace identifier |
| `execution_duration` | Total wall-clock time in milliseconds |
| `status` | `OK`, `ERROR`, or `IN_PROGRESS` |
| `request` | Serialised input to the root span |
| `response` | Serialised output of the root span |
| `spans` | List of all child spans with timing and I/O |

The `spans` column allows you to drill into individual tool calls and LLM invocations without opening the UI — useful for automated quality checks or cost accounting.

In [ ]:
import pandas as pd

# ── Retrieve the most recent traces from this experiment ──────────────────────
traces_df = mlflow.search_traces(
    experiment_names=[EXPERIMENT_PATH],
    max_results=10,
    order_by=["timestamp_ms DESC"],
)

print(f"Traces retrieved: {len(traces_df)}")
print()

# ── Display summary columns ───────────────────────────────────────────────────
summary_cols = ["request_id", "execution_duration", "status", "request", "response"]
available    = [c for c in summary_cols if c in traces_df.columns]
display(traces_df[available])

# ── Drill into spans for the most recent trace ────────────────────────────────
if not traces_df.empty and "spans" in traces_df.columns:
    latest_trace = traces_df.iloc[0]
    print(f"\nSpans for trace {latest_trace['request_id']}:")
    print(f"  Total duration : {latest_trace['execution_duration']:.0f} ms")
    print(f"  Status         : {latest_trace['status']}")
    print()

    spans = latest_trace["spans"]
    for span in spans:
        span_name     = span.get("name", "unknown")
        span_duration = span.get("end_time_ns", 0) - span.get("start_time_ns", 0)
        span_ms       = span_duration / 1_000_000
        span_status   = span.get("status", {}).get("status_code", "OK")
        print(f"  Span: {span_name:<40} {span_ms:>8.0f} ms  [{span_status}]")

## D. Tag Experiments for Reproducibility

An MLflow **run** is reproducible only if you can reconstruct the exact configuration that produced it. **Tags** are the mechanism for capturing configuration metadata alongside a run's logged metrics and artifacts.

The recommended tag set for a RAG agent covers every component that affects output quality:

| Tag | Why It Matters |
|---|---|
| `agent_version` | Identifies which code version was running |
| `llm_endpoint` | Different models produce different outputs |
| `embedding_model` | Determines which vectors are in the index |
| `chunk_size` | Affects context window usage and recall |
| `chunk_overlap` | Affects boundary coverage for retrieval |
| `vector_search_type` | `ann` vs `hybrid` changes retrieval behaviour |

Tags differ from **params**: params are logged once per run and cannot be changed; tags are mutable and can be updated after a run completes. Use params for hyperparameters and tags for configuration labels and version identifiers.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration constants used for tagging
# Change these values and re-run to simulate a new agent version.
# ---------------------------------------------------------------------------
EMBEDDING_MODEL     = "databricks-gte-large-en"
CHUNK_SIZE          = 512
CHUNK_OVERLAP       = 64
VECTOR_SEARCH_TYPE  = "hybrid"

# ── Start a tracked run with reproducibility tags ─────────────────────────────
with mlflow.start_run(run_name=f"arxiv-agent-{AGENT_VERSION}") as run:

    # Tags capture the full configuration needed to reproduce this run
    mlflow.set_tags({
        "agent_version"      : AGENT_VERSION,
        "llm_endpoint"       : LLM_ENDPOINT,
        "embedding_model"    : EMBEDDING_MODEL,
        "chunk_size"         : str(CHUNK_SIZE),
        "chunk_overlap"      : str(CHUNK_OVERLAP),
        "vector_search_type" : VECTOR_SEARCH_TYPE,
    })

    # Log params for numeric/hyperparameter values
    mlflow.log_params({
        "llm_endpoint"       : LLM_ENDPOINT,
        "chunk_size"         : CHUNK_SIZE,
        "chunk_overlap"      : CHUNK_OVERLAP,
        "vector_search_type" : VECTOR_SEARCH_TYPE,
        "num_tools"          : len(all_tools),
    })

    # Run an evaluation query inside the tracked run so its trace is linked
    eval_query  = "Explain the difference between full fine-tuning and LoRA."
    eval_result = agent_executor.invoke({"input": eval_query})

    # Log a simple quality metric (replace with LLM-judge score in Lab 08)
    answer_length = len(eval_result["output"])
    mlflow.log_metric("answer_length_chars", answer_length)

    # Log the agent model artifact
    mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model=None,   # placeholder — full model logging covered in Lab 05
        pip_requirements=[
            "databricks-sdk", "databricks-vectorsearch",
            "mlflow", "langchain", "langchain-community",
            "unitycatalog-langchain", "databricks-agents",
        ],
    )

    run_id_v1 = run.info.run_id

print(f"Run ID        : {run_id_v1}")
print(f"Agent version : {AGENT_VERSION}")
print(f"Answer length : {answer_length} chars")
print()
print("Tags written:")
for k, v in {
    "agent_version": AGENT_VERSION,
    "llm_endpoint": LLM_ENDPOINT,
    "embedding_model": EMBEDDING_MODEL,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "vector_search_type": VECTOR_SEARCH_TYPE,
}.items():
    print(f"  {k:<25}: {v}")

## E. Compare Agent Versions

`mlflow.search_runs()` queries the run store using a SQL-like filter string. Filtering by `tags.agent_version` lets you pull every run for a specific version and compare metrics across versions — the foundation of **systematic agent evaluation** before Lab 08.

The comparison workflow is:

1. Tag every run with `agent_version` at log time (Section D).
2. Query all runs for a given version with `mlflow.search_runs(filter_string="tags.agent_version = 'v1'")`.
3. Aggregate metrics (mean, p50, p95 latency; mean answer length; pass rate) across the version's runs.
4. Compare aggregated tables across versions to decide which configuration to promote.

In Lab 08 you will replace `answer_length_chars` with LLM-judge scores (faithfulness, relevance, correctness). The comparison pattern is identical.

In [ ]:
import pandas as pd

# ── Simulate a second agent version with a different configuration ─────────────
# In a real workflow you would change the LLM, chunk size, or embedding model.
# Here we change only the version label and chunk_size to illustrate the pattern.
AGENT_VERSION_V2   = "v2"
CHUNK_SIZE_V2      = 256   # smaller chunks — may improve precision, reduce recall
CHUNK_OVERLAP_V2   = 32

with mlflow.start_run(run_name=f"arxiv-agent-{AGENT_VERSION_V2}") as run_v2:
    mlflow.set_tags({
        "agent_version"      : AGENT_VERSION_V2,
        "llm_endpoint"       : LLM_ENDPOINT,
        "embedding_model"    : EMBEDDING_MODEL,
        "chunk_size"         : str(CHUNK_SIZE_V2),
        "chunk_overlap"      : str(CHUNK_OVERLAP_V2),
        "vector_search_type" : VECTOR_SEARCH_TYPE,
    })
    mlflow.log_params({
        "llm_endpoint"       : LLM_ENDPOINT,
        "chunk_size"         : CHUNK_SIZE_V2,
        "chunk_overlap"      : CHUNK_OVERLAP_V2,
        "vector_search_type" : VECTOR_SEARCH_TYPE,
        "num_tools"          : len(all_tools),
    })
    result_v2       = agent_executor.invoke({"input": eval_query})
    answer_length_v2 = len(result_v2["output"])
    mlflow.log_metric("answer_length_chars", answer_length_v2)
    run_id_v2 = run_v2.info.run_id

print(f"v2 run logged: {run_id_v2}")

# ── Search and compare runs across both versions ──────────────────────────────
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_PATH).experiment_id

runs_df = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.agent_version IN ('v1', 'v2')",
    order_by=["start_time DESC"],
)

# ── Build a readable comparison table ─────────────────────────────────────────
compare_cols = [
    "run_id",
    "tags.agent_version",
    "params.chunk_size",
    "params.chunk_overlap",
    "params.vector_search_type",
    "metrics.answer_length_chars",
    "status",
]
available_cols = [c for c in compare_cols if c in runs_df.columns]
comparison_df  = runs_df[available_cols].rename(
    columns={
        "tags.agent_version"         : "agent_version",
        "params.chunk_size"          : "chunk_size",
        "params.chunk_overlap"       : "chunk_overlap",
        "params.vector_search_type"  : "vs_type",
        "metrics.answer_length_chars": "answer_length_chars",
    }
)

print("\nAgent version comparison:")
display(comparison_df)

# ── Print delta summary ───────────────────────────────────────────────────────
if len(comparison_df) >= 2:
    delta = answer_length_v2 - answer_length
    print(f"\nAnswer length delta (v2 vs v1): {delta:+d} chars")
    print("Replace answer_length_chars with LLM-judge scores in Lab 08.")

## Key Takeaways

| Concept | What You Did |
|---|---|
| **Autologging** | Called `mlflow.langchain.autolog(log_traces=True)` to instrument every agent invocation without modifying agent code |
| **Traces** | Generated traces for three distinct query types; inspected span trees in the MLflow Experiment UI |
| **Spans** | Drilled into individual LLM call, tool call, and retrieval spans programmatically via `mlflow.search_traces()` |
| **Experiment Tags** | Used `mlflow.set_tags()` inside `mlflow.start_run()` to capture the full configuration (6 tag keys) for each run |
| **Run Reproducibility** | Tagged runs with `agent_version`, making every configuration recoverable and auditable |
| **Version Comparison** | Used `mlflow.search_runs(filter_string="tags.agent_version IN (...)")` to build a side-by-side comparison table |

### Exam Tips

- `mlflow.langchain.autolog(log_traces=True)` is the **single call** needed to enable tracing — no manual span creation is required.
- A **trace** is the entire end-to-end record of one agent invocation. A **span** is one unit of work within a trace (e.g. a single LLM call or tool invocation). Know this distinction for the exam.
- `mlflow.search_traces()` returns a DataFrame; `mlflow.search_runs()` also returns a DataFrame but covers logged runs, not individual traces.
- **Tags vs params**: params are write-once hyperparameters; tags are mutable labels. Use tags for version identifiers and configuration flags; use params for numeric experiment knobs.
- `mlflow.set_experiment()` with a personal `/Users/<username>/...` path avoids permission collisions in shared workspaces — a common production pattern.

---

**Next:** Lab 08 (Evaluation with LLM-as-Judge) replaces `answer_length_chars` with faithfulness, relevance, and correctness scores produced by an LLM judge. The tagging and `search_runs` comparison pattern you built here is reused directly.

## Architecture Diagram

![Architecture Diagram](../assets/diagrams/lab-06-tracing-reproducible-agents.png)

## Key Concepts

| Concept | Definition |
|---|---|
| **Trace** | The complete end-to-end record of one agent invocation — a tree of spans with a single root span representing the top-level call |
| **Span** | One unit of work within a trace: an LLM call, a tool invocation, or a retrieval step, each with start time, end time, inputs, and outputs |
| **Autologging** | `mlflow.langchain.autolog(log_traces=True)` — a single call that instruments all LangChain chains and agents to emit traces without code changes |
| **MLflow Experiment** | A named container for runs and traces, identified by a workspace path; set with `mlflow.set_experiment()` |
| **Experiment Tags** | Mutable key-value metadata attached to a run via `mlflow.set_tags()`; queryable with `filter_string` in `mlflow.search_runs()` |
| **Run Reproducibility** | The ability to reconstruct the exact configuration that produced a run; achieved by tagging every run with the complete set of environment and configuration variables |
| **search_traces** | `mlflow.search_traces(experiment_names=[...])` — returns a Pandas DataFrame of traces with columns for `request_id`, `execution_duration`, `status`, `request`, `response`, and `spans` |

## Exam Preparation

### Exam Practice Questions

**Q1.** A developer wants to automatically capture LLM call latency, tool inputs/outputs, and retrieval results for every agent invocation without modifying the agent code. What is the correct single call to enable this?

- A) `mlflow.start_run(log_traces=True)`
- B) `mlflow.langchain.autolog(log_traces=True)`
- C) `mlflow.enable_tracing(framework="langchain")`
- D) `mlflow.log_trace(agent_executor)`

**Answer: B** — `mlflow.langchain.autolog(log_traces=True)` is the correct single-call enablement. Option A is not valid syntax; `start_run` has no `log_traces` parameter. Option C is not a real MLflow API. Option D is not a real MLflow API.

---

**Q2.** What is the difference between a **trace** and a **span** in the context of MLflow agent observability?

- A) A trace is for single-step chains; a span is for multi-step agents
- B) A trace is the complete end-to-end record of one agent invocation; a span is one unit of work (e.g. one LLM call or tool call) within that trace
- C) A trace records only LLM calls; a span records only tool calls
- D) A trace and a span are synonyms — MLflow uses both terms for the same object

**Answer: B** — A trace is the full tree rooted at the top-level agent invocation. A span is one leaf or branch of that tree, representing a single operation with its own start time, end time, and I/O. The distinction maps directly to OpenTelemetry terminology, which the exam references.

---

**Q3.** A team has logged 50 agent runs, each tagged with `agent_version`. They want to retrieve only the runs where `agent_version = "v3"`. Which `mlflow.search_runs()` call is correct?

- A) `mlflow.search_runs(filter_string="params.agent_version = 'v3'")`
- B) `mlflow.search_runs(filter_string="attributes.agent_version = 'v3'")`
- C) `mlflow.search_runs(filter_string="tags.agent_version = 'v3'")`
- D) `mlflow.search_runs(tag_name="agent_version", tag_value="v3")`

**Answer: C** — Tags are accessed with the `tags.` prefix in the filter string. Params use `params.`, metrics use `metrics.`, and run attributes use `attributes.`. Option D is not valid syntax for `search_runs`.

---

**Q4.** An autologged LangChain agent produces a trace with three child spans: one labelled `ChatDatabricks`, one labelled `search_arxiv_papers`, and one labelled `similarity_search`. Which span types do these represent?

- A) LLM call, tool call, retrieval
- B) Tool call, LLM call, retrieval
- C) Retrieval, tool call, LLM call
- D) All three are LLM call spans

**Answer: A** — `ChatDatabricks` is the LangChain LLM class, so its span represents the LLM call. `search_arxiv_papers` is the `@tool`-decorated function, so its span represents the tool call. `similarity_search` is the Vector Search SDK call inside the tool, so its span represents the retrieval step.

---

**Q5.** A developer uses `mlflow.log_params()` to capture `chunk_size=512` and `mlflow.set_tags()` to capture `agent_version="v1"`. After the run completes, a reviewer finds a typo in the version label and wants to correct it to `"v1.0"`. Which statement is correct?

- A) Both params and tags can be updated after the run; use `mlflow.log_param()` and `mlflow.set_tag()` respectively
- B) Neither params nor tags can be updated after a run completes
- C) Tags can be updated after the run using `mlflow.set_tag(run_id, key, value)`; params cannot be updated once logged
- D) Params can be updated after the run using `mlflow.log_param()`; tags cannot be updated

**Answer: C** — MLflow params are **immutable** once logged — attempting to re-log the same param key raises an error. Tags are **mutable** and can be updated at any time using `mlflow.set_tag(run_id=..., key=..., value=...)`. This is a deliberate design: params record experimental inputs (immutable for reproducibility) and tags record descriptive metadata (mutable for correction and categorisation).

## Cost Breakdown

| Component | Detail | Estimated Cost |
|---|---|---|
| Databricks Serverless compute | Notebook execution (~25 min DBU) | ~$0.50 |
| LLM token usage | 3 traced queries + 2 version evaluation calls | ~$0.50 |
| Vector Search queries | Retrieval spans during RESEARCH-type queries | Included in serverless |
| **Total** | | **~$1** |

**Estimated time:** ~25 min | **Estimated cost:** ~$1

> Costs vary by workspace region and current DBU pricing. Use the Databricks Cost Dashboard to track actuals.